# Exercise 4: E-Commerce Text & Quantitative Analytics

**Objective:** Perform group-level statistical aggregation and unstructured text visualization on real-world customer review data.

**Dataset:** Flipkart Product Customer Reviews Dataset (Kaggle)

This notebook:
1. Loads the Flipkart reviews CSV into a Pandas DataFrame.
2. Cleans the rating and price fields.
3. Groups reviews by product name and calculates mean rating and mean price.
4. Creates and displays a Word Cloud from the `Summary` column.
5. Saves the cleaned data and product-level summary as CSV files.

> **Before running:** Download the Flipkart Product Customer Reviews Dataset from Kaggle and place its CSV file in the same folder as this notebook, or upload the CSV when prompted below. The notebook attempts to identify the relevant columns even if their capitalization or spacing differs.

## 1. Install and import required libraries

Run the cell below once in your Jupyter environment. If you are using Google Colab, the same notebook will also work there.

In [ ]:
%pip install -q pandas numpy matplotlib wordcloud

In [ ]:
import re
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud, STOPWORDS

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

print("Libraries imported successfully.")

## 2. Load the dataset

**Option A (recommended):** Put the downloaded `.csv` file in the same directory as this notebook.  
**Option B:** If no CSV is found, the next cell lets you upload one in a Jupyter environment that supports `ipywidgets`/file upload. You can also set `DATA_PATH` manually.

In [ ]:
# If your CSV is in another folder, enter its full or relative path here.
# Example: DATA_PATH = "data/flipkart_reviews.csv"
DATA_PATH = None

def locate_csv():
    if DATA_PATH:
        path = Path(DATA_PATH)
        if not path.exists():
            raise FileNotFoundError(f"CSV file not found: {path}")
        return path

    candidates = []
    for pattern in ("*.csv", "**/*.csv"):
        candidates.extend(Path(".").glob(pattern))
    # Remove duplicate paths and common unrelated notebook export files.
    candidates = sorted({p for p in candidates if p.is_file() and not p.name.startswith("._")})
    if candidates:
        print("CSV files found:")
        for i, p in enumerate(candidates, start=1):
            print(f"  {i}. {p}")
        if len(candidates) == 1:
            return candidates[0]
        # Prefer filenames that look related to Flipkart/reviews.
        preferred = [p for p in candidates if any(k in p.name.lower() for k in ("flipkart", "review", "product"))]
        if len(preferred) == 1:
            return preferred[0]
        print("\nMore than one CSV was found. Set DATA_PATH in this cell to the correct file.")
        raise RuntimeError("Please set DATA_PATH to the dataset CSV.")
    raise FileNotFoundError(
        "No CSV file found. Download the Kaggle dataset and place its CSV beside this notebook, "
        "then rerun this cell and the following cells."
    )

csv_path = locate_csv()
print(f"Loading: {csv_path}")

# Try common encodings because public CSV datasets may not all use UTF-8.
for encoding in ("utf-8", "utf-8-sig", "latin-1"):
    try:
        df = pd.read_csv(csv_path, encoding=encoding)
        print(f"Dataset loaded using {encoding!r} encoding.")
        break
    except UnicodeDecodeError:
        continue
else:
    raise UnicodeError("Could not decode the CSV. Try opening and saving it as UTF-8.")

print("Dataset shape:", df.shape)
display(df.head())

## 3. Inspect and identify the required columns

The exercise requires a product name, rating, price, and review summary. This cell normalizes column names and maps common dataset variants to those fields. If your CSV uses unusual column names, edit the `COLUMN_ALIASES` dictionary or set the mapping manually.

In [ ]:
print("Original columns:")
print(df.columns.tolist())

def normalize_column(name):
    return re.sub(r"[^a-z0-9]+", "", str(name).strip().lower())

normalized_to_original = {normalize_column(c): c for c in df.columns}

COLUMN_ALIASES = {
    "product_name": [
        "productname", "product", "producttitle", "title", "name"
    ],
    "rating": [
        "rating", "rate", "ratings", "reviewrating", "customer rating"
    ],
    "price": [
        "price", "productprice", "sellingprice", "discountedprice", "amount"
    ],
    "summary": [
        "summary", "review", "reviewtext", "reviewsummary", "comment", "comments"
    ],
}

def find_column(field):
    for alias in COLUMN_ALIASES[field]:
        key = normalize_column(alias)
        if key in normalized_to_original:
            return normalized_to_original[key]
    return None

column_map = {field: find_column(field) for field in COLUMN_ALIASES}
print("\nDetected column mapping:")
for field, col in column_map.items():
    print(f"{field:>12}: {col}")

missing = [field for field in ("product_name", "rating", "price", "summary") if column_map[field] is None]
if missing:
    raise ValueError(
        "Could not identify required column(s): "
        + ", ".join(missing)
        + ". Inspect df.columns and update COLUMN_ALIASES in this cell."
    )

## 4. Clean the rating and price fields

Ratings and prices may contain text such as `4.2 out of 5`, `₹ 1,299`, commas, currency symbols, or blank values. The following code extracts numeric values, converts them to numbers, and removes rows with missing product names, ratings, or prices for the product-level summary.

The original review text is retained for the Word Cloud. We do not assume a particular rating scale; inspect the printed range after cleaning.

In [ ]:
# Work on a copy and rename the required columns to consistent names.
data = pd.DataFrame({
    "product_name": df[column_map["product_name"]],
    "rating_raw": df[column_map["rating"]],
    "price_raw": df[column_map["price"]],
    "summary": df[column_map["summary"]],
})

def extract_number(value):
    if pd.isna(value):
        return np.nan
    # Keep digits, decimal point, and minus sign; remove commas/currency/text.
    text = str(value).replace(",", "")
    match = re.search(r"-?\d+(?:\.\d+)?", text)
    return float(match.group()) if match else np.nan

data["rating"] = data["rating_raw"].apply(extract_number)
data["price"] = data["price_raw"].apply(extract_number)

data["product_name"] = data["product_name"].astype("string").str.strip()
data["summary"] = data["summary"].fillna("").astype(str).str.strip()

# Treat empty strings and common placeholders as missing.
data["product_name"] = data["product_name"].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
data["summary"] = data["summary"].replace({"nan": "", "None": ""})

print("Missing values after conversion:")
display(data[["product_name", "rating", "price", "summary"]].isna().sum().to_frame("missing_count"))

# Rows with missing product/rating/price cannot contribute to both requested means.
analysis_data = data.dropna(subset=["product_name", "rating", "price"]).copy()

print(f"Rows in original dataset: {len(data):,}")
print(f"Rows usable for product summary: {len(analysis_data):,}")
print(f"Rows excluded from product summary: {len(data) - len(analysis_data):,}")
print(f"Rating range: {analysis_data['rating'].min()} to {analysis_data['rating'].max()}")
print(f"Price range: {analysis_data['price'].min()} to {analysis_data['price'].max()}")

display(analysis_data.head())

## 5. Group by product name and calculate mean rating and mean price

This performs the required group-level aggregation. The resulting summary table is stored in a new DataFrame named **`items`**.

In [ ]:
items = (
    analysis_data
    .groupby("product_name", as_index=False)
    .agg(
        mean_rating=("rating", "mean"),
        mean_price=("price", "mean"),
        review_count=("product_name", "size")
    )
    .sort_values(by="product_name")
    .reset_index(drop=True)
)

# Round means for readable display while retaining numeric values.
items["mean_rating"] = items["mean_rating"].round(2)
items["mean_price"] = items["mean_price"].round(2)

print("Product-level summary (items):")
display(items.head(20))
print(f"Number of unique products: {len(items):,}")

## 6. Visualize mean rating and mean price

The plots below provide a quick overview of the product-level summary. If the dataset contains many products, the charts show the 15 products with the most reviews so that labels remain readable.

In [ ]:
top_products = (
    items.sort_values("review_count", ascending=False)
    .head(15)
    .sort_values("mean_rating", ascending=True)
)

if not top_products.empty:
    plt.figure(figsize=(11, 6))
    plt.barh(top_products["product_name"].astype(str), top_products["mean_rating"])
    plt.xlabel("Mean rating")
    plt.ylabel("Product")
    plt.title("Mean customer rating — 15 most-reviewed products")
    plt.tight_layout()
    plt.show()

    price_products = (
        items.sort_values("review_count", ascending=False)
        .head(15)
        .sort_values("mean_price", ascending=True)
    )
    plt.figure(figsize=(11, 6))
    plt.barh(price_products["product_name"].astype(str), price_products["mean_price"])
    plt.xlabel("Mean price")
    plt.ylabel("Product")
    plt.title("Mean product price — 15 most-reviewed products")
    plt.tight_layout()
    plt.show()
else:
    print("No product groups are available to plot.")

## 7. Generate a Word Cloud from the `Summary` column

The Word Cloud highlights frequently occurring words in customer feedback. The code combines the review summaries, removes common English stop words, and displays the most frequent terms.

In [ ]:
review_text = " ".join(
    data.loc[data["summary"].str.strip().ne(""), "summary"].astype(str).tolist()
)

if review_text.strip():
    custom_stopwords = set(STOPWORDS)
    # Optional additions for generic words that may not carry useful review meaning.
    custom_stopwords.update({"product", "flipkart", "one", "also", "good"})

    wordcloud = WordCloud(
        width=1200,
        height=600,
        background_color="white",
        stopwords=custom_stopwords,
        collocations=False,
        max_words=150,
        contour_width=0
    ).generate(review_text)

    plt.figure(figsize=(15, 7))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title("Most Frequent Customer Feedback Terms", fontsize=16)
    plt.tight_layout(pad=0)
    plt.show()
else:
    print("The Summary column contains no usable text, so a Word Cloud cannot be generated.")

## 8. Display the most frequent feedback terms

This optional text-frequency table complements the Word Cloud by listing common terms and their counts.

In [ ]:
from collections import Counter

if review_text.strip():
    tokens = re.findall(r"\b[a-zA-Z]{2,}\b", review_text.lower())
    stop_words = set(STOPWORDS)
    stop_words.update({"product", "flipkart", "one", "also", "good"})
    word_counts = Counter(word for word in tokens if word not in stop_words)

    frequent_terms = pd.DataFrame(
        word_counts.most_common(20),
        columns=["word", "frequency"]
    )
    display(frequent_terms)
else:
    frequent_terms = pd.DataFrame(columns=["word", "frequency"])
    print("No review text available.")

## 9. Save output files

The notebook saves:
- `items_product_summary.csv` — product-wise mean rating, mean price, and review count.
- `cleaned_flipkart_reviews.csv` — cleaned review data used for analysis.

These output files can be committed to GitHub if your course rules allow sharing the dataset. Check the dataset's license and avoid uploading data if redistribution is restricted.

In [ ]:
items.to_csv("items_product_summary.csv", index=False)
analysis_data[["product_name", "rating", "price", "summary"]].to_csv(
    "cleaned_flipkart_reviews.csv", index=False
)

print("Saved:")
print(" - items_product_summary.csv")
print(" - cleaned_flipkart_reviews.csv")

## 10. Conclusion

In this exercise, the Flipkart customer review dataset was loaded into a Pandas DataFrame. The rating and price columns were converted into numeric values, and incomplete rows were excluded from the product-level aggregation. Reviews were grouped by product name to calculate the mean rating and mean price, with the resulting table stored in `items`. Finally, a Word Cloud was generated from the `Summary` column to visualize frequently used customer feedback terms.

**Required outputs:** `items` DataFrame and the customer-feedback Word Cloud.